# 🧠 K-Nearest Neighbors (KNN) — Explained Notebook

**Objective:** Predict whether a social-network user purchases a product (`Purchased = 1`) or not (`Purchased = 0`) using **Age** and **EstimatedSalary**.

**How KNN works:** For each new customer, calculate the distance to existing training customers, select the **K closest**, and predict the class that receives the most votes. This is called **lazy learning** because KNN stores training examples rather than learning a separate equation during training.

**In this notebook:** `K = 5` and the distance is **Euclidean distance**. Run cells from top to bottom.

**This notebook focuses on KNN classification and numerical evaluation; no visualizations or plots are included.**


## 📦 Step 1: Importing libraries

- **NumPy:** numerical operations and array manipulation.
- **Pandas:** reading the CSV file and selecting columns.


In [ ]:
# Import NumPy for working with arrays and numerical computations.
import numpy as np
# Import Pandas to read the CSV dataset into a table (DataFrame).
import pandas as pd


## 📂 Step 2: Load the dataset

This dataset contains `User ID`, `Gender`, `Age`, `EstimatedSalary`, and `Purchased`.

- **X (features):** columns at index `2` and `3`, i.e. **Age** and **EstimatedSalary**.
- **y (target):** last column, **Purchased** (`0` = No, `1` = Yes).
- **Example:** A customer aged 35 with estimated salary 20,000 is represented as `[35, 20000]`; the model predicts 0 or 1.

The CSV name below matches the supplied file. Keep the CSV in the same folder as the notebook.


In [ ]:
# Read the CSV supplied with this notebook. File should be in the current folder.
dataset = pd.read_csv('Social_Network_Ads(1).csv')
# iloc[:, [2, 3]] selects ALL rows and columns 2 (Age) and 3 (EstimatedSalary).
# .values converts the selected Pandas columns into a NumPy array.
X = dataset.iloc[:, [2, 3]].values
# iloc[:, -1] selects the LAST column (Purchased): 0 = No, 1 = Yes.
y = dataset.iloc[:, -1].values


## ✂️ Step 3: Split training and test data

We train on **75%** of the rows and evaluate on the remaining **25%**.

- `test_size=0.25`: one quarter becomes the test set.
- `random_state=0`: makes the random split reproducible.
- We keep the test labels (`y_test`) aside so we can check how well the model predicts unseen examples.


In [ ]:
# Import a utility to split the examples into training and testing groups.
from sklearn.model_selection import train_test_split
# 75% of customers are used for training; 25% are reserved for testing.
# random_state=0 keeps the split the same every time we run the notebook.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)


## ⚖️ Step 4: Standardize the features

KNN uses **distance**, so a large-valued feature such as salary can dominate a small-valued feature such as age. Standardization puts both on comparable scales:

**z = (value − training mean) / training standard deviation**

- `fit_transform(X_train)`: learns the mean and standard deviation **only from training data** and scales training rows.
- `transform(X_test)`: applies those same statistics to the test data, avoiding **data leakage**.

*Example:* Without scaling, a salary difference of 10,000 could overpower an age difference of 10.


In [ ]:
# StandardScaler converts each feature to its z-score using training statistics.
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
# Learn each training column's mean/std and scale the training features.
X_train = sc.fit_transform(X_train)
# Scale test features using training mean/std (never fit again on test data).
X_test = sc.transform(X_test)


## 🎯 Step 5: Train the KNN classifier

- `n_neighbors=5`: look at the **five nearest training examples** for each prediction.
- `metric='minkowski', p=2`: Minkowski distance with `p=2` is **Euclidean distance**.
- `fit(X_train, y_train)`: stores the scaled training examples and their labels.

**Euclidean distance between two customers:**

`distance = sqrt((age_1 - age_2)^2 + (salary_1 - salary_2)^2)`

Here, distance is computed on **scaled features**. If 3 of the 5 neighbors bought a product and 2 did not, KNN predicts **Purchased = 1**.


In [ ]:
# Import K-Nearest Neighbors classifier from scikit-learn.
from sklearn.neighbors import KNeighborsClassifier
# n_neighbors=5: use 5 closest customers to make a majority-vote prediction.
# 'minkowski' distance with p=2 means Euclidean (straight-line) distance.
classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
# Store the scaled training examples and their purchased/not-purchased labels.
classifier.fit(X_train, y_train)


## 🔮 Step 6: Predict outcomes for test customers

For each row in `X_test`, KNN finds its five closest training neighbors and assigns the majority class. `y_pred` contains the predicted 0/1 values in the same row order as `y_test`.


In [ ]:
# Predict Purchased (0 or 1) for every customer in the held-out test set.
# For each row, KNN finds its five nearest training rows and takes a majority vote.
y_pred = classifier.predict(X_test)


## ✅ Step 7: Evaluate using a confusion matrix

For classes `0` (Not Purchased) and `1` (Purchased), the matrix is arranged as:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| **Actual 0** | True Negative (TN) | False Positive (FP) |
| **Actual 1** | False Negative (FN) | True Positive (TP) |

**Accuracy = (TP + TN) / (TP + TN + FP + FN)**. A confusion matrix reveals *which kinds of mistakes* the classifier makes, not just how many predictions were correct.


In [ ]:
# Import the confusion-matrix function to compare actual and predicted labels.
from sklearn.metrics import confusion_matrix
# Rows = actual labels; columns = predicted labels, ordered [0, 1].
# Thus cm[0,0]=TN, cm[0,1]=FP, cm[1,0]=FN, cm[1,1]=TP.
cm = confusion_matrix(y_test, y_pred)
# Print the 2x2 matrix so we can inspect correct and incorrect classifications.
print(cm)
